# Evaluación Parcial N°3 - Google Play Store

## 🎯 Objetivo
Consolidar en un único notebook el trabajo analítico de la Evaluación 2 y ampliarlo con los componentes de la Evaluación 3: documentación técnica reproducible, validación del pipeline de datos, visualizaciones avanzadas y un dashboard interactivo construido con Dash.

**Pregunta de trabajo:** ¿qué patrones del ecosistema Google Play permiten describir, segmentar y predecir si una aplicación será gratuita o de pago, y cómo comunicar estos resultados mediante un dashboard profesional?

## 🧱 Estructura esperada del proyecto

```text
proyecto-googleplay-ev3/
│
├── data/
│   └── raw/
│       └── googleplaystore.csv
│
├── docs/
│   └── README.md
│
└── Evaluacion_3_GooglePlay_Pipeline.ipynb
```

Este notebook está pensado para funcionar dentro de esa estructura mínima, sin Docker, sin API propia y usando únicamente las librerías autorizadas.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import dash
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)

## 🔍 Exploración y Diagnóstico (EDA)

Primero se carga el dataset original y se audita su estructura. Esta fase es crítica porque el archivo de Google Play suele contener variables numéricas almacenadas como texto, valores faltantes y duplicados por aplicación.

In [ ]:
from pathlib import Path

possible_paths = [
    Path("data/raw/googleplaystore.csv"),
    Path("/content/proyecto-googleplay-ev3/data/raw/googleplaystore.csv"),
    Path("googleplaystore.csv"),
    Path("googleplaystore-2.csv")
]

for path in possible_paths:
    if path.exists():
        data_path = path
        break
else:
    raise FileNotFoundError("No se encontró googleplaystore.csv en las rutas esperadas.")


df_original = pd.read_csv(data_path)
df = df_original.copy()

print(f"Ruta utilizada: {data_path}")
print(f"Shape original: {df.shape}")
display(df.head())
display(df.info())

In [ ]:
resumen_nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "dtype": df.dtypes.astype(str)
}).sort_values("nulos", ascending=False)

print("Duplicados por App:", df.duplicated(subset=["App"]).sum())
display(resumen_nulos)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["Rating"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribución de Rating")
axes[0].set_xlabel("Rating")

sns.boxplot(x=df["Rating"], ax=axes[1], color="salmon")
axes[1].set_title("Boxplot de Rating")
axes[1].set_xlabel("Rating")

plt.tight_layout()
plt.show()

**Justificación EDA:** se inspecciona `Rating` con histograma y boxplot porque es una variable continua central para el análisis de calidad percibida, y además permite evaluar asimetría y outliers antes de definir la estrategia de limpieza.

## 🛠️ Preprocesamiento

Se aplica una estrategia de limpieza orientada a reproducibilidad y robustez. La mediana se utiliza para imputar `Rating` porque es menos sensible a valores extremos; las columnas numéricas almacenadas como texto se convierten con reglas explícitas; y se eliminan registros con nulos críticos que afectarían el modelado.

In [ ]:
# 1. Eliminar duplicados por nombre de aplicación
df = df.drop_duplicates(subset=["App"]).copy()

# 2. Imputar Rating con mediana
df["Rating"] = df["Rating"].fillna(df["Rating"].median())

# 3. Conversión de columnas numéricas

def parse_installs(value: str) -> float:
    value = str(value).replace(",", "").replace("+", "").strip()
    return pd.to_numeric(value, errors="coerce")


def parse_price(value: str) -> float:
    value = str(value).replace("$", "").strip()
    return pd.to_numeric(value, errors="coerce")


def parse_size(value: str) -> float:
    value = str(value).strip()
    if value == "Varies with device":
        return np.nan
    if value.endswith("M"):
        return float(value[:-1]) * 1_000_000
    if value.endswith("k"):
        return float(value[:-1]) * 1_000
    return np.nan


df["Installs"] = df["Installs"].apply(parse_installs)
df["Price"] = df["Price"].apply(parse_price)
df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce")
df["Size"] = df["Size"].astype(str).apply(parse_size)
df["Last Updated"] = pd.to_datetime(df["Last Updated"], errors="coerce")

# 4. Imputación robusta de Size con mediana
df["Size"] = df["Size"].fillna(df["Size"].median())

# 5. Normalización de texto
df["Category"] = df["Category"].astype(str).str.strip().str.upper()
df["Type"] = df["Type"].astype(str).str.strip()

# 6. Eliminación de nulos críticos
critical_cols = ["Installs", "Price", "Reviews", "Type", "Content Rating", "Current Ver", "Android Ver"]
df = df.dropna(subset=critical_cols).copy()

# 7. Regla de consistencia
mask_inconsistent = (df["Type"].eq("Free")) & (df["Price"] > 0)
df = df.loc[~mask_inconsistent].copy()

print("Shape luego de limpieza básica:", df.shape)
display(df.head())

In [ ]:
# Detección de outliers en Rating por IQR
q1 = df["Rating"].quantile(0.25)
q3 = df["Rating"].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

before_rows = len(df)
df = df[(df["Rating"] >= lower) & (df["Rating"] <= upper)].copy()
after_rows = len(df)

print(f"Filas eliminadas por IQR en Rating: {before_rows - after_rows}")
print(f"Límites IQR: [{lower:.2f}, {upper:.2f}]")
print(f"Shape final preprocesado: {df.shape}")

## 📊 Análisis descriptivo posterior a la limpieza

Con el dataset limpio, se revisan distribuciones clave y relaciones entre variables. Esta etapa permite verificar si las transformaciones preservaron el sentido de negocio de los datos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, y="Category", order=df["Category"].value_counts().head(10).index, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Top 10 categorías por frecuencia")
axes[0, 0].set_xlabel("Cantidad")
axes[0, 0].set_ylabel("Category")

sns.countplot(data=df, x="Type", ax=axes[0, 1], color="seagreen")
axes[0, 1].set_title("Distribución de Type")
axes[0, 1].set_xlabel("Type")
axes[0, 1].set_ylabel("Cantidad")

sns.scatterplot(data=df.sample(min(len(df), 3000), random_state=RANDOM_STATE), x="Installs", y="Reviews", hue="Type", alpha=0.6, ax=axes[1, 0])
axes[1, 0].set_xscale("log")
axes[1, 0].set_yscale("log")
axes[1, 0].set_title("Installs vs Reviews (escala log)")

sns.boxplot(data=df, x="Type", y="Rating", ax=axes[1, 1], palette="colorblind")
axes[1, 1].set_title("Rating por tipo de app")

plt.tight_layout()
plt.show()

In [ ]:
corr_df = df[["Rating", "Reviews", "Size", "Installs", "Price"]].copy()
correlation = corr_df.corr(method="spearman")

display(correlation.round(3))

plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlación de Spearman entre variables numéricas")
plt.show()

**Conclusión EDA:** `Installs` y `Reviews` tienden a relacionarse fuertemente, lo que es esperable porque mayor base instalada suele generar más reseñas. `Price` presenta una distribución muy sesgada hacia cero, por lo que su interpretación debe hacerse con cuidado en visualizaciones y modelos.

## 🛠️ Ingeniería de variables

Para machine learning, se construye una variable objetivo binaria y se generan predictores numéricos y categóricos. Se usa OneHotEncoder para evitar introducir relaciones ordinales artificiales en `Category` y `Content Rating`.

In [ ]:
df["Type_encoded"] = (df["Type"] == "Paid").astype(int)

feature_cols_num = ["Rating", "Reviews", "Size", "Installs", "Price"]
feature_cols_cat = ["Category", "Content Rating"]
target_col = "Type_encoded"

X = df[feature_cols_num + feature_cols_cat].copy()
y = df[target_col].copy()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, feature_cols_num),
    ("cat", categorical_transformer, feature_cols_cat)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(y.value_counts(normalize=True).rename("proportion"))

## 🤖 Modelado

Se proponen tres enfoques con distinta complejidad: una Regresión Logística como baseline interpretable, Random Forest como modelo no lineal robusto y Gradient Boosting como alternativa más expresiva. Todos se encapsulan en pipelines para evitar data leakage.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200, class_weight="balanced"))
    ]),
    "Gradient Boosting": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred

    results.append({
        "Modelo": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "AUC_ROC": roc_auc_score(y_test, y_proba)
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False)
display(results_df.round(4))

In [ ]:
best_model_name = results_df.iloc[0]["Modelo"]
best_model = fitted_models[best_model_name]

y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1] if hasattr(best_model, 'predict_proba') else y_pred_best

cm = confusion_matrix(y_test, y_pred_best)
cm_df = pd.DataFrame(cm, index=["Real_Free", "Real_Paid"], columns=["Pred_Free", "Pred_Paid"])

display(cm_df)
print(best_model_name)
print(classification_report(y_test, y_pred_best, target_names=["Free", "Paid"]))

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results_df.melt(id_vars="Modelo", var_name="Métrica", value_name="Valor"), x="Modelo", y="Valor", hue="Métrica")
plt.title("Comparación de métricas por modelo")
plt.xticks(rotation=15)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 📊 Clustering

Como complemento no supervisado, se segmentan aplicaciones según popularidad y tamaño. Se usa K-Means porque las variables numéricas principales permiten una geometría razonablemente compacta tras escalar y aplicar transformaciones logarítmicas a las magnitudes más sesgadas.

In [ ]:
cluster_df = df[["Rating", "Reviews", "Size", "Installs"]].copy()
cluster_df["Reviews"] = np.log1p(cluster_df["Reviews"])
cluster_df["Installs"] = np.log1p(cluster_df["Installs"])

scaler_cluster = StandardScaler()
X_cluster = scaler_cluster.fit_transform(cluster_df)

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
clusters = kmeans.fit_predict(X_cluster)
df["Cluster"] = clusters

sil = silhouette_score(X_cluster, clusters)
print(f"Silhouette Score: {sil:.4f}")

display(df.groupby("Cluster")[["Rating", "Reviews", "Size", "Installs", "Price"]].mean().round(2))

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
cluster_proj = pca.fit_transform(X_cluster)
plot_df = pd.DataFrame({
    "PC1": cluster_proj[:, 0],
    "PC2": cluster_proj[:, 1],
    "Cluster": df["Cluster"].astype(str),
    "Type": df["Type"].values
})

fig = px.scatter(
    plot_df.sample(min(len(plot_df), 4000), random_state=RANDOM_STATE),
    x="PC1", y="PC2", color="Cluster", symbol="Type",
    title="Clusters de apps proyectados con PCA"
)
fig.update_layout(template="plotly_white")
fig.show()

## 📋 Evaluación e integridad del dataset

Además del modelado, la Evaluación 3 necesita evidencias claras de calidad del proceso. Por eso se incorpora una auditoría de integridad con reglas de negocio simples y verificables.

In [ ]:
def auditar_dataset(df_input: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df_input.columns:
        rows.append({
            "Columna": col,
            "Tipo": str(df_input[col].dtype),
            "Nulos": int(df_input[col].isna().sum()),
            "% Nulos": round(df_input[col].isna().mean() * 100, 2),
            "Únicos": int(df_input[col].nunique(dropna=True)),
            "Estado": "OK" if df_input[col].isna().sum() == 0 else "REVISAR"
        })
    return pd.DataFrame(rows)


def validar_rangos(df_input: pd.DataFrame) -> pd.DataFrame:
    checks = [
        ("Rating entre 1 y 5", df_input["Rating"].between(1, 5)),
        ("Price no negativo", df_input["Price"] >= 0),
        ("Installs no negativo", df_input["Installs"] >= 0),
        ("Reviews no negativo", df_input["Reviews"] >= 0),
    ]
    rows = []
    total = len(df_input)
    for name, condition in checks:
        ok = int(condition.sum())
        fail = int(total - ok)
        rows.append({
            "Regla": name,
            "Cumplen": ok,
            "Violan": fail,
            "% Cumplimiento": round(ok / total * 100, 2),
            "Estado": "PASS" if fail == 0 else "FAIL"
        })
    return pd.DataFrame(rows)


audit_report = auditar_dataset(df)
range_report = validar_rangos(df)
consistency_report = df.loc[(df["Type"] == "Free") & (df["Price"] > 0), ["App", "Type", "Price"]]

display(audit_report)
display(range_report)
display(consistency_report.head())

integrity_score = (range_report["Estado"] == "PASS").mean() * 100
print(f"Score general de integridad: {integrity_score:.1f}")

## 📈 Dashboard interactivo con Dash

El dashboard resume KPIs, distribuciones y filtros de negocio. Se implementa dentro del mismo notebook para cumplir el requisito de un único archivo entregable, pero manteniendo una estructura clara y ejecutable.

In [ ]:
# Dataset agregado para dashboard
category_summary = (
    df.groupby("Category", as_index=False)
      .agg(apps=("App", "count"),
           avg_rating=("Rating", "mean"),
           avg_installs=("Installs", "mean"),
           avg_price=("Price", "mean"))
      .sort_values("apps", ascending=False)
)

kpi_total_apps = int(len(df))
kpi_paid_share = round((df["Type"] == "Paid").mean() * 100, 2)
kpi_avg_rating = round(df["Rating"].mean(), 2)
kpi_total_reviews = int(df["Reviews"].sum())

print(category_summary.head())
print(kpi_total_apps, kpi_paid_share, kpi_avg_rating, kpi_total_reviews)

In [ ]:
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Dashboard Google Play Store"),
    html.P("Resumen ejecutivo y técnico del pipeline analítico."),

    html.Div([
        html.Div([html.H3("Apps"), html.H2(f"{kpi_total_apps:,}")], style={"padding": "10px", "border": "1px solid #ddd", "borderRadius": "8px", "width": "23%"}),
        html.Div([html.H3("% Paid"), html.H2(f"{kpi_paid_share}%")], style={"padding": "10px", "border": "1px solid #ddd", "borderRadius": "8px", "width": "23%"}),
        html.Div([html.H3("Rating medio"), html.H2(f"{kpi_avg_rating}")], style={"padding": "10px", "border": "1px solid #ddd", "borderRadius": "8px", "width": "23%"}),
        html.Div([html.H3("Reviews totales"), html.H2(f"{kpi_total_reviews:,}")], style={"padding": "10px", "border": "1px solid #ddd", "borderRadius": "8px", "width": "23%"}),
    ], style={"display": "flex", "justifyContent": "space-between", "gap": "10px", "marginBottom": "20px"}),

    html.Div([
        html.Label("Filtrar por tipo de app"),
        dcc.Dropdown(
            id="type_filter",
            options=[{"label": t, "value": t} for t in sorted(df["Type"].dropna().unique())],
            value="Free",
            clearable=False
        )
    ], style={"marginBottom": "20px", "width": "40%"}),

    dcc.Graph(id="bar_categories"),
    dcc.Graph(id="scatter_installs_reviews"),
    dcc.Graph(id="box_rating_type"),
    dash_table.DataTable(
        id="table_summary",
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left", "padding": "8px"},
        style_header={"fontWeight": "bold"}
    )
], style={"padding": "20px", "fontFamily": "Arial"})

@app.callback(
    Output("bar_categories", "figure"),
    Output("scatter_installs_reviews", "figure"),
    Output("box_rating_type", "figure"),
    Output("table_summary", "data"),
    Output("table_summary", "columns"),
    Input("type_filter", "value")
)
def update_dashboard(selected_type):
    filtered = df[df["Type"] == selected_type].copy()

    cat_plot = (
        filtered.groupby("Category", as_index=False)
                .agg(apps=("App", "count"))
                .sort_values("apps", ascending=False)
                .head(10)
    )

    fig_bar = px.bar(cat_plot, x="apps", y="Category", orientation="h", title=f"Top categorías - {selected_type}")
    fig_bar.update_layout(template="plotly_white", yaxis={"categoryorder": "total ascending"})

    sample_scatter = filtered.sample(min(len(filtered), 2500), random_state=RANDOM_STATE)
    fig_scatter = px.scatter(
        sample_scatter,
        x="Installs", y="Reviews", color="Category",
        hover_data=["App", "Rating", "Price"],
        title=f"Installs vs Reviews - {selected_type}",
        log_x=True, log_y=True
    )
    fig_scatter.update_layout(template="plotly_white")

    fig_box = px.box(filtered, x="Content Rating", y="Rating", color="Content Rating", title=f"Rating por Content Rating - {selected_type}")
    fig_box.update_layout(template="plotly_white")

    table_df = (
        filtered.groupby("Category", as_index=False)
                .agg(apps=("App", "count"), avg_rating=("Rating", "mean"), avg_price=("Price", "mean"))
                .sort_values("apps", ascending=False)
                .head(15)
                .round(2)
    )

    return (
        fig_bar,
        fig_scatter,
        fig_box,
        table_df.to_dict("records"),
        [{"name": col, "id": col} for col in table_df.columns]
    )

# Para ejecutar localmente o en Colab/Jupyter:
# app.run(debug=False, port=8050)

### Instrucción de uso del dashboard

Descomenta la última línea `app.run(...)` para levantar la aplicación. En Google Colab puede requerirse un túnel o ejecución local; en Jupyter local bastará con abrir la URL `http://127.0.0.1:8050/`.

## 📝 Informe Final

### Hallazgos clave
- El ecosistema está fuertemente dominado por apps gratuitas, lo que introduce desbalance de clases en la predicción.
- Las variables de popularidad, especialmente `Installs` y `Reviews`, concentran gran parte de la señal analítica.
- El clustering permite distinguir perfiles de apps masivas, nicho y tamaño intermedio.

### Decisiones técnicas justificadas
- Se utilizó imputación por mediana en variables sesgadas.
- Se usó OneHotEncoder para variables categóricas nominales.
- Se encapsuló todo en un solo notebook para cumplir el formato de entrega.
- Dash se integró dentro del notebook para mantener una solución reproducible y profesional.

### Resultados de negocio
- El modelo supervisado entrega una base para apoyar decisiones de monetización (`Free` vs `Paid`).
- El dashboard facilita comunicar resultados a audiencias ejecutivas y técnicas con filtros y KPIs.

### Próximos pasos
- Incorporar ingeniería de atributos temporales desde `Last Updated`.
- Evaluar técnicas para clases desbalanceadas.
- Desplegar el dashboard en un servicio web cuando el curso lo permita.

## ⚠️ Consideraciones

- El dataset contiene sesgos estructurales del marketplace y no representa causalidad directa.
- La calidad del modelo depende de variables limitadas; no se usan descripciones textuales ni señales externas.
- Dash está incluido en el notebook como evidencia funcional, pero su ejecución final depende del entorno donde se abra el archivo.
- No se utiliza Docker ni API REST, porque el alcance del proyecto se ajusta a `data/`, `docs/` y un notebook principal único.